# Extracting surprises without a preview

Previews started on 3 Aug 2026: none of the 827 events before then has one, about 11% of later Q3 events lack one, and 13 of the first 21 Q4 events do. The live agent needs an extraction that works from the call facts alone. `extraction.extract_event(..., preview=False)` uses a separate prompt (`prompts.EXTRACTION_SYSTEM_NO_PREVIEW`): it allows only expectations the call itself states (consensus, prior guidance, targets) and records `expectation_source` as `call` or `none`. The preview prompt is unchanged, so every existing extraction stays valid.

**Part A: hide the preview on the train events.** Re-extract the 1,076 train events without their preview and compare with the preview extractions on the same events: how much ΔR² is lost, and how a preview-trained model does on no-preview features.

**Part B: the real no-preview events.** Extract the 997 Q3 events that never had a preview, split them by time, and choose how to score them: a model trained on them, a model trained on Part A's hidden-preview features, the preview-trained model, or the plain `net_w_first3` score. Test is scored once, at the end.

Test events of the preview split are never touched. Classification uses the frozen v2 taxonomy, as before.

Cost of the full run: about 1,076 + 997 extraction calls (`gpt-6-luna`, medium) and about 400 assignment calls of 40 surprises.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from ml import metrics, model, preprocess
from suprise_dim import data, dataset, extraction, features, grouping
from suprise_dim.prompts import EXTRACTION_SYSTEM_NO_PREVIEW, build_user_message_no_preview

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

FROZEN = "v2"
A_EXTRACT, A_ASSIGN = "train_nopreview", "v2_train_nopreview"  # data/surprise/train_nopreview_extractions.jsonl
B_EXTRACT, B_ASSIGN = "np_q3", "v2_np_q3"                      # data/surprise/np_q3_extractions.jsonl
ES = model.ES

split = data.load_split()
tax = grouping.load_taxonomy(FROZEN)
preview_events = data.load_eligible_events()
np_events = data.load_no_preview_events("2026Q3")
events_by_id = {e["event_id"]: e for e in preview_events + np_events}
train_events = [events_by_id[eid] for eid in split["train"]]
surprises_preview = pd.read_csv(dataset.surprises_path(FROZEN))

if data.NO_PREVIEW_SPLIT_PATH.exists():
    np_split = data.load_split(data.NO_PREVIEW_SPLIT_PATH)
else:
    np_split = data.make_no_preview_split(np_events, train_frac=0.8)
    data.save_split(np_split, data.NO_PREVIEW_SPLIT_PATH)
print(f"train (preview split) {len(train_events)} | no-preview events {len(np_events)}: "
      f"train {len(np_split['train'])}, test {len(np_split['test'])} (from {np_split['test_start']})")

train (preview split) 1076 | no-preview events 997: train 793, test 204 (from 2026-07-31T13:00:00+00:00)


In [2]:
# Helpers shared by both parts.
def extract_and_assign(events, extract_name, assign_name, n=None):
    events = events[:n] if n else events
    extraction.run_extraction(events, out_path=data.extractions_path(extract_name), preview=False)
    ids = {e["event_id"] for e in events}
    records = [r for r in extraction.load_extractions(data.extractions_path(extract_name)) if r["event_id"] in ids]
    grouping.assign_items(grouping.surprise_items(records), tax, assign_name)


def complete_ids(extract_name, assign_name, wanted):
    """Events extracted whose every surprise is assigned (events with no surprise count as complete)."""
    records = [r for r in extraction.load_extractions(data.extractions_path(extract_name)) if r["event_id"] in wanted]
    assigned = set(grouping.load_assignments(assign_name)["item_id"])
    return {r["event_id"] for r in records
            if all(f"{r['event_id']}#{i}" in assigned for i in range(len(r["surprises"])))}


def event_features(surprises, labels):
    """Feature rows for the events in `labels` (event id -> "train"/"test")."""
    split_like = {"train": [e for e, l in labels.items() if l == "train"], "test": [e for e, l in labels.items() if l == "test"]}
    return features.build_event_features(surprises, split_like, events_by_id, tax).reset_index(drop=True)


def best_config(frame):
    grid = model.grid_search(frame, preprocess.feature_sets(frame.columns), np.logspace(-1, 4, 11))
    return grid, grid.iloc[0]


def fit_best(frame, best):
    return model.fit(frame, preprocess.feature_sets(frame.columns)[best["feature_set"]], best["alpha"],
                     target_mode=best["target_mode"], log_counts=bool(best["log_counts"]))

## 1. Inspect the no-preview prompt

In [3]:
sample = train_events[0]
print(EXTRACTION_SYSTEM_NO_PREVIEW)
print("\n" + "=" * 80 + "\n")
print(build_user_message_no_preview(sample))

You compare an earnings release against what was expected before it.

You receive facts from the earnings call, written after the release. No research preview is available, so the only expectations you can use are the ones the call itself states: analyst consensus or estimates the call mentions, the company's prior guidance or targets, or what management says it or the market expected.

List every distinct way the reported results, guidance, or commentary differ from those stated expectations. Cover everything the facts allow: headline numbers, guidance, segments, margins, operating metrics, capital allocation, one-off items, and management tone.

Rules:
- One item per distinct surprise. Do not merge unrelated surprises; do not split one into many.
- Describe each surprise concretely in plain language. Do not label it with an abstract category; the categories will be derived later from many releases.
- A surprise needs a stated expectation. Growth or a change against the prior year or 

In [4]:
# One live call: the same event with and without its preview.
no_preview = extraction.extract_event(sample, preview=False)
with_preview = surprises_preview[surprises_preview["event_id"] == sample["event_id"]].sort_values("position")
print(f"WITH preview ({len(with_preview)}):")
for r in with_preview.itertuples():
    print(f"  [{r.direction}/{r.magnitude}/{r.expectation_source}] {r.description[:160]}")
print(f"\nWITHOUT preview ({len(no_preview.surprises)}):")
for s in no_preview.surprises:
    print(f"  [{s.direction}/{s.magnitude}/{s.expectation_source}] {s.description[:160]}")

WITH preview (8):
  [positive/small/preview] Q3 revenue was $118.4 million, about $3.9 million above the $114.55 million consensus and above the company’s $114–115 million guidance. The preview described t
  [positive/moderate/preview] Twist raised FY2026 revenue guidance to $456–457 million, about $12 million above the prior midpoint. The preview said consensus was sitting near the middle of 
  [positive/moderate/preview] Q4 revenue guidance of $123–124 million is well above the preview’s implied $113.6–117.6 million, which suggested flat-to-modestly-up sequential revenue. The ne
  [positive/large/preview] Academic and government revenue sharply rebounded despite the preview’s expectation of a weak research-spending backdrop: it rose 32% year over year and 21% seq
  [negative/small/preview] Q3 adjusted EBITDA loss was about $11.3 million, worse than the preview’s base-case range of roughly $8–10 million in losses. Management nevertheless reaffirmed
  [positive/moderate/none] Managemen

## Part A. Hide the preview on the train events

### A1. Extract and classify

`N_A` counts events. Start small, then `None` for all 1,076. Reruns resume.

In [6]:
N_A = None  # None for every train event
extract_and_assign(train_events, A_EXTRACT, A_ASSIGN, N_A)

Extracting 1066 events (10 already done) with gpt-6-luna, no-preview prompt...
[1/1066] ea_ALSN_Q2_2026: 1 surprises
[2/1066] ea_AVA_Q2_2026: 0 surprises
[3/1066] ea_ADEA_Q2_2026: 1 surprises
[4/1066] ea_PLOW_Q2_2026: 1 surprises
[5/1066] ea_AEIS_Q2_2026: 3 surprises
[6/1066] ea_AMRC_Q2_2026: 1 surprises
[7/1066] ea_VRTX_Q2_2026: 2 surprises
[8/1066] ea_MATX_Q2_2026: 1 surprises
[9/1066] ea_INSP_Q2_2026: 0 surprises
[10/1066] ea_PAY_Q2_2026: 2 surprises
[11/1066] ea_SNAP_Q2_2026: 2 surprises
[12/1066] ea_BWXT_Q2_2026: 4 surprises
[13/1066] ea_PLTR_Q2_2026: 3 surprises
[14/1066] ea_ARVN_Q2_2026: 2 surprises
[15/1066] ea_CIFR_Q2_2026: 2 surprises
[16/1066] ea_RVTY_Q2_2026: 4 surprises
[17/1066] ea_DOCN_Q2_2026: 2 surprises
[18/1066] ea_CBT_Q3_2026: 4 surprises
[19/1066] ea_DD_Q2_2026: 3 surprises
[20/1066] ea_EVEX_Q2_2026: 2 surprises
[21/1066] ea_SGC_Q2_2026: 0 surprises
[22/1066] ea_HSIC_Q2_2026: 3 surprises
[23/1066] ea_IT_Q2_2026: 1 surprises
[24/1066] ea_SHLS_Q2_2026: 0 surprises
[2

### A2. Surprise lists with and without the preview (same events)

In [7]:
a_ids = complete_ids(A_EXTRACT, A_ASSIGN, set(split["train"]))
surprises_hidden = dataset.build_extraction_frame(A_EXTRACT, A_ASSIGN, {e: "train" for e in a_ids})
surprises_shown = surprises_preview[surprises_preview["event_id"].isin(a_ids)]

def describe(s, n_events):
    return pd.Series({
        "surprises per event": len(s) / n_events,
        "events with no surprise": n_events - s["event_id"].nunique(),
        **{f"direction {k}": v for k, v in s["direction"].value_counts(normalize=True).round(3).items()},
        **{f"source {k}": v for k, v in s["expectation_source"].value_counts(normalize=True).round(3).items()},
    })

print(f"{len(a_ids)} events compared")
display(pd.DataFrame({"with preview": describe(surprises_shown, len(a_ids)), "without preview": describe(surprises_hidden, len(a_ids))}).fillna(0).round(3))
pd.DataFrame({
    "with preview": surprises_shown["dimension"].value_counts(normalize=True),
    "without preview": surprises_hidden["dimension"].value_counts(normalize=True),
}).fillna(0).round(3)

1071 events compared


,with preview,without preview
direction mixed,0.112,0.037
direction negative,0.285,0.266
direction positive,0.587,0.693
direction unclear,0.016,0.004
events with no surprise,0.000,333.000
source call,0.008,0.911
source none,0.176,0.089
source preview,0.816,0.000
surprises per event,7.755,1.536


,with preview,without preview
dimension,,
accounting_tax_special_items,0.023,0.008
bottom_line_earnings,0.067,0.030
capital_allocation,0.039,0.026
cash_balance_sheet_liquidity,0.064,0.022
commercial_economics,0.056,0.019
costs_expenses,0.034,0.016
development_execution_regulatory,0.054,0.057
disclosure_delivery,0.004,0.000
external_environment,0.022,0.013


### A3. How much is lost without the preview?

Event features from both extractions of the same train events. Single scores need no fitting; ridge is chosen by cross-validation on each feature version. **Transfer** fits on preview features and predicts the same events from their no-preview features (fold by fold, out of fold), which is what happens if the current preview-trained model meets a no-preview event.

In [8]:
labels_a = {e: "train" for e in a_ids}
feat_shown = event_features(surprises_shown, labels_a)
feat_hidden = event_features(surprises_hidden, labels_a)
assert (feat_shown["event_id"] == feat_hidden["event_id"]).all()

y, es = feat_shown["target_pct"], feat_shown[ES]
singles = {
    "net_w_first3 with preview": model.rank_to_percentile(feat_shown["net_w_first3"]),
    "net_w_first3 without preview": model.rank_to_percentile(feat_hidden["net_w_first3"]),
    "net_w with preview": model.rank_to_percentile(feat_shown["net_w"]),
    "net_w without preview": model.rank_to_percentile(feat_hidden["net_w"]),
}
display(metrics.evaluate(y, singles, es).round(4))
metrics.compare_to(metrics.paired_bootstrap(y, singles, es, n_boot=2000), "net_w_first3 without preview").round(4)

,n,r2_base,r2_full,delta_r2,spearman
net_w with preview,1023.0,0.0725,0.2391,0.1665,0.4657
net_w_first3 with preview,1023.0,0.0725,0.2351,0.1625,0.4653
net_w without preview,1023.0,0.0725,0.1345,0.0619,0.2937
net_w_first3 without preview,1023.0,0.0725,0.1336,0.0611,0.2916


,mean_diff,ci_low,ci_high,P(net_w_first3 without preview better)
net_w_first3 with preview,-0.1015,-0.1399,-0.0629,0.0000
net_w with preview,-0.1057,-0.1437,-0.0673,0.0000
net_w without preview,-0.0009,-0.0048,0.0026,0.3075


In [9]:
grid_shown, best_shown = best_config(feat_shown)
grid_hidden, best_hidden = best_config(feat_hidden)
cols = ["feature_set", "alpha", "target_mode", "log_counts", "oof_delta_r2", "oof_spearman"]
print("best with preview:   ", best_shown[cols].to_dict())
print("best without preview:", best_hidden[cols].to_dict())

def oof(frame, best, predict_on=None):
    out, _ = model.out_of_fold(frame, preprocess.feature_sets(frame.columns)[best["feature_set"]], best["alpha"],
                               target_mode=best["target_mode"], log_counts=bool(best["log_counts"]), predict_on=predict_on)
    return out.set_index("event_id")["oof"]

ridge = pd.DataFrame({
    "ridge with preview": oof(feat_shown, best_shown),
    "ridge without preview": oof(feat_hidden, best_hidden),
    "transfer: preview-trained on no-preview features": oof(feat_shown, best_shown, predict_on=feat_hidden),
}).dropna()
rows = feat_shown.set_index("event_id").loc[ridge.index]
preds = {c: ridge[c].to_numpy() for c in ridge}
display(metrics.evaluate(rows["target_pct"], preds, rows[ES]).round(4))
metrics.compare_to(metrics.paired_bootstrap(rows["target_pct"], preds, rows[ES], n_boot=2000), "ridge without preview").round(4)

best with preview:    {'feature_set': 'all', 'alpha': 1000.0, 'target_mode': 'raw', 'log_counts': False, 'oof_delta_r2': 0.185103298200749, 'oof_spearman': 0.49154592316872137}
best without preview: {'feature_set': 'all', 'alpha': 10000.0, 'target_mode': 'residual', 'log_counts': True, 'oof_delta_r2': 0.07930966643189796, 'oof_spearman': 0.31260457264340025}


,n,r2_base,r2_full,delta_r2,spearman
ridge with preview,842.0,0.0762,0.2613,0.1851,0.4915
ridge without preview,842.0,0.0762,0.1555,0.0793,0.3126
transfer: preview-trained on no-preview features,842.0,0.0762,0.1466,0.0703,0.2997


,mean_diff,ci_low,ci_high,P(ridge without preview better)
ridge with preview,-0.1059,-0.1497,-0.0627,0.0000
transfer: preview-trained on no-preview features,0.0088,-0.0063,0.0239,0.8715


**Reading Part A.** The gap between "with" and "without preview" is the cost of a missing preview. If "transfer" is clearly below "ridge without preview", a preview-trained model should not score no-preview events; use a model trained on no-preview features instead.

## Part B. The real no-preview events

### B1. Which events they are

Mostly July, before previews existed, so they skew toward early reporters.

In [10]:
np_frame = pd.DataFrame({"event_id": [e["event_id"] for e in np_events],
                         "date": pd.to_datetime([e["event_datetime"] for e in np_events])})
np_frame["set"] = np.where(np_frame["event_id"].isin(np_split["test"]), "test", "train")
np_frame.groupby([np_frame["date"].dt.to_period("M"), "set"]).size().unstack(fill_value=0)

/var/folders/qb/pd0pzd3x0jj2c_2nkn2hmrxw0000gn/T/ipykernel_30294/1406459861.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  np_frame.groupby([np_frame["date"].dt.to_period("M"), "set"]).size().unstack(fill_value=0)


set,test,train
date,,
2026-07,34,793
2026-08,137,0
2026-09,33,0


### B2. Extract and classify

`N_B` counts events: start small, then `None` for all 997.

In [11]:
N_B = None  # None for every no-preview event
extract_and_assign(np_events, B_EXTRACT, B_ASSIGN, N_B)

Extracting 997 events (0 already done) with gpt-6-luna, no-preview prompt...
[1/997] ea_BSET_Q2_2026: 0 surprises
[2/997] ea_LNN_Q3_2026: 0 surprises
[3/997] ea_GIS_Q4_2026: 0 surprises
[4/997] ea_FDS_Q3_2026: 0 surprises
[5/997] ea_GBX_Q3_2026: 0 surprises
[6/997] ea_CULP_Q4_2026: 1 surprises
[7/997] ea_PSMT_Q3_2026: 0 surprises
[8/997] ea_FC_Q3_2026: 1 surprises
[9/997] ea_RXST_Q2_2026: 2 surprises
[10/997] ea_KRUS_Q3_2026: 3 surprises
[11/997] ea_MSM_Q3_2026: 3 surprises
[12/997] ea_EPAC_Q3_2026: 2 surprises
[13/997] ea_PCYO_Q3_2026: 2 surprises
[14/997] ea_PENG_Q3_2026: 2 surprises
[15/997] ea_NTIC_Q3_2026: 0 surprises
[16/997] ea_AZZ_Q1_2027: 2 surprises
[17/997] ea_SMPL_Q3_2026: 1 surprises
[18/997] ea_BYRN_Q2_2026: 1 surprises
[19/997] ea_PEP_Q2_2026: 3 surprises
[20/997] ea_ANGO_Q4_2026: 0 surprises
[21/997] ea_EDUC_Q1_2027: 0 surprises
[22/997] ea_CRMT_Q4_2026: 0 surprises
[23/997] ea_LEVI_Q2_2026: 3 surprises
[24/997] ea_DAL_Q2_2026: 1 surprises
[25/997] ea_FAST_Q2_2026: 1 su

In [12]:
labels_b = {e: "train" for e in np_split["train"]} | {e: "test" for e in np_split["test"]}
b_ids = complete_ids(B_EXTRACT, B_ASSIGN, set(labels_b))
surprises_np = dataset.build_extraction_frame(B_EXTRACT, B_ASSIGN, {e: labels_b[e] for e in b_ids})
feat_np = event_features(surprises_np, {e: labels_b[e] for e in b_ids})
np_train = feat_np[feat_np["split"] == "train"].reset_index(drop=True)
np_test = feat_np[feat_np["split"] == "test"].reset_index(drop=True)
print(f"complete: {len(b_ids)} of {len(labels_b)} | train {len(np_train)} | test {len(np_test)}")
display(describe(surprises_np, len(b_ids)).to_frame("real no-preview events").round(3))

complete: 996 of 997 | train 793 | test 203


,real no-preview events
surprises per event,1.781
events with no surprise,234.000
direction positive,0.717
direction negative,0.249
direction mixed,0.024
direction unclear,0.010
source call,0.915
source none,0.085


### B3. Choose how to score no-preview events (no-preview train only)

All four candidates are scored on the same no-preview train events:

- **np-trained ridge**: chosen and evaluated by cross-validation on these events (out of fold).
- **hidden-preview ridge**: fitted on all of Part A's no-preview features (train events of the preview split, a different set of events), then applied here.
- **preview ridge**: fitted on the preview features of the same Part A events, then applied here (the transfer case on real events).
- **net_w_first3**: the plain score, no fitting.

In [13]:
grid_np, best_np = best_config(np_train)
print("best np-trained:", best_np[cols].to_dict())

oof_np = oof(np_train, best_np)
cand = pd.DataFrame({"np-trained ridge": oof_np}, index=oof_np.index)
cand["hidden-preview ridge"] = fit_best(feat_hidden, best_hidden).predict(np_train.set_index("event_id").loc[cand.index].reset_index())
cand["preview ridge"] = fit_best(feat_shown, best_shown).predict(np_train.set_index("event_id").loc[cand.index].reset_index())
cand["net_w_first3"] = model.rank_to_percentile(np_train.set_index("event_id").loc[cand.index, "net_w_first3"])
cand = cand.dropna()  # the first time block is never validated for the np-trained ridge
rows = np_train.set_index("event_id").loc[cand.index]
preds = {c: cand[c].to_numpy() for c in cand}
b3 = metrics.evaluate(rows["target_pct"], preds, rows[ES])
display(b3.round(4))
metrics.compare_to(metrics.paired_bootstrap(rows["target_pct"], preds, rows[ES], n_boot=2000), "np-trained ridge").round(4)

best np-trained: {'feature_set': 'totals', 'alpha': 100.0, 'target_mode': 'raw', 'log_counts': True, 'oof_delta_r2': 0.011488838371629617, 'oof_spearman': 0.1294631555987795}


,n,r2_base,r2_full,delta_r2,spearman
hidden-preview ridge,649.0,0.1043,0.1228,0.0185,0.1690
preview ridge,649.0,0.1043,0.1189,0.0146,0.1701
net_w_first3,649.0,0.1043,0.1166,0.0123,0.1724
np-trained ridge,649.0,0.1043,0.1158,0.0115,0.1295


,mean_diff,ci_low,ci_high,P(np-trained ridge better)
hidden-preview ridge,-0.0069,-0.0207,0.0043,0.1105
preview ridge,-0.0030,-0.0142,0.0070,0.2840
net_w_first3,-0.0007,-0.0124,0.0121,0.4535


### B4. Test, once

The candidate with the highest ΔR² in B3 among those ranking in the right direction (positive Spearman) is fitted on its training data and scored on the no-preview test events, next to the competition baselines. ΔR² alone ignores the sign of a prediction, so a model that ranks backwards could otherwise win. Run this cell once, after B3 is final.

In [14]:
eligible = b3[b3["spearman"] > 0]
assert len(eligible), "no candidate ranks in the right direction on no-preview train"
CHOICE = eligible["delta_r2"].idxmax()
print("chosen:", CHOICE)
test_rows = np_test.copy()
if CHOICE == "np-trained ridge":
    test_pred = fit_best(np_train, best_np).predict(test_rows)
elif CHOICE == "hidden-preview ridge":
    test_pred = fit_best(feat_hidden, best_hidden).predict(test_rows)
elif CHOICE == "preview ridge":
    test_pred = fit_best(feat_shown, best_shown).predict(test_rows)
else:
    test_pred = model.rank_to_percentile(test_rows["net_w_first3"])

BASELINES = sorted(c for c in test_rows.columns if c.startswith("baseline__"))
preds = {CHOICE: test_pred, "net_w_first3": model.rank_to_percentile(test_rows["net_w_first3"])}
preds |= {b.split("__")[1]: test_rows[b].to_numpy() for b in BASELINES}
common = test_rows[[ES] + BASELINES].notna().all(axis=1).to_numpy()
print(f"test events: {len(test_rows)} | with the ranked surprise and all baselines: {common.sum()}")
display(metrics.evaluate(test_rows["target_pct"][common], {k: v[common] for k, v in preds.items()}, test_rows[ES][common]).round(4))
samples = metrics.paired_bootstrap(test_rows["target_pct"][common], {k: v[common] for k, v in preds.items()}, test_rows[ES][common], n_boot=2000)
metrics.compare_to(samples, CHOICE).round(4)

chosen: hidden-preview ridge
test events: 203 | with the ranked surprise and all baselines: 180


,n,r2_base,r2_full,delta_r2,spearman
z-ai,180.0,0.0573,0.1392,0.0820,0.2672
hidden-preview ridge,180.0,0.0573,0.1310,0.0737,0.2791
moonshotai,180.0,0.0573,0.1272,0.0700,0.2769
net_w_first3,180.0,0.0573,0.1199,0.0626,0.2857
gemini,180.0,0.0573,0.1160,0.0587,0.2693
openai,180.0,0.0573,0.0984,0.0411,0.1787
deepseek,180.0,0.0573,0.0899,0.0327,0.2123


,mean_diff,ci_low,ci_high,P(hidden-preview ridge better)
net_w_first3,0.0100,-0.0176,0.0351,0.7845
deepseek,0.0399,-0.0180,0.1093,0.9190
gemini,0.0153,-0.0515,0.0797,0.6815
moonshotai,0.0044,-0.0662,0.0734,0.5420
openai,0.0332,-0.0176,0.0953,0.9000
z-ai,-0.0075,-0.0787,0.0628,0.4060
